# 05d — Independent residual-CNN transfer confirmation

This notebook is a **validation-only method-development checkpoint**. The untouched test split is not used.

Scientific purpose: test whether the SRM-teacher-derived local risk $D_i$ transfers to a detector that was **not used to construct $D_i$**.

Protocol locked before looking at the CNN transfer result:

1. The SRM teacher and local risk model come from `05c` and are not refit here.
2. The CNN is trained **only on train-split covers and random-allocation stegos** at the teacher payload.
3. CNN hyperparameters are taken unchanged from `config/experiment.yaml`; no validation-based epoch tuning is performed here.
4. Allocation transfer is evaluated on the **second half of the validation split**, disjoint from the first 750 validation images used for the `05c` alpha diagnostic.
5. All alpha values are scored on the **same common-feasible source images**.
6. Primary transfer endpoint: paired difference in CNN score change, candidate alpha minus predictability-only $\alpha=1$. A 95% bootstrap CI fully below zero is the confirmatory criterion.
7. This notebook does **not** overwrite `frozen_allocator.json`.


In [ ]:
from pathlib import Path
import gc, hashlib, json, joblib, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdhlab.io import read_gray
from rdhlab.pipeline import load_payload_freeze, run_frozen_image_precomputed, deterministic_seed
from rdhlab.blockcodec import analyze_blocks
from rdhlab.allocation import rank_blocks
from rdhlab.detectors import (
    train_residual_cnn, score_residual_cnn, save_cnn,
    detector_metrics, paired_detector_bootstrap,
)
from rdhlab.transfer import bootstrap_location_ci, paired_method_bootstrap

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
seed=int(config['project']['seed'])
bs=int(config['dataset']['block_size'])
fixed_fpr=float(config['detectors']['fixed_fpr'])
confidence=float(config['statistics']['confidence'])
n_boot=int(config['statistics']['cluster_bootstrap_resamples'])
batch_size=int(config['detectors']['cnn_batch_size'])
epochs=int(config['detectors']['cnn_epochs'])

manifest=pd.read_csv(config['dataset']['prepared_manifest'])
train=manifest[manifest.split=='train'].reset_index(drop=True)
val=manifest[manifest.split=='validation'].reset_index(drop=True)

out=Path('/workspace/results/cnn_transfer')
out.mkdir(parents=True,exist_ok=True)
models=Path('/workspace/results/models')
models.mkdir(parents=True,exist_ok=True)

local_risk=joblib.load(models/'srm_teacher_local_risk.joblib')
alpha_summary=pd.read_csv('/workspace/results/srm_teacher_risk/alpha_validation_summary.csv')
teacher_validation=json.loads(Path('/workspace/results/srm_teacher_risk/teacher_validation.json').read_text())
teacher_bpp=float(teacher_validation['payload_bpp'])
alpha_grid=sorted(map(float,config['allocator']['alpha_grid']))

print('Teacher payload:',teacher_bpp)
print('Alpha grid:',alpha_grid)
print('Train:',len(train),'Validation:',len(val))


## Candidate alpha fixed from 05c before CNN transfer

The candidate is selected from the already observed SRM-teacher trade-off, not from the CNN result.

Rule: among positive $\alpha<1$, choose the **smallest** alpha that preserves at least 90% of the maximum teacher-logit reduction relative to predictability-only $\alpha=1$, while improving PSNR relative to pure detectability allocation $\alpha=0$.

This rule treats the CNN as a confirmation detector rather than another alpha-tuning instrument.


In [ ]:
s=alpha_summary.sort_values('alpha').copy()
r0=s.loc[np.isclose(s.alpha,0.0)].iloc[0]
r1=s.loc[np.isclose(s.alpha,1.0)].iloc[0]
max_teacher_gain=float(r1.teacher_delta_median-r0.teacher_delta_median)
if max_teacher_gain <= 0:
    raise RuntimeError('05c does not show lower teacher response at alpha=0 than alpha=1.')

s['teacher_gain_retained']=(float(r1.teacher_delta_median)-s.teacher_delta_median)/max_teacher_gain
eligible=s[(s.alpha>0) & (s.alpha<1) & (s.teacher_gain_retained>=0.90) & (s.psnr_mean>float(r0.psnr_mean))]
if len(eligible)==0:
    raise RuntimeError('No positive alpha satisfies the pre-CNN candidate rule; redesign before transfer testing.')

candidate_alpha=float(eligible.sort_values('alpha').iloc[0].alpha)
rule={
    'candidate_alpha':candidate_alpha,
    'teacher_payload_bpp':teacher_bpp,
    'rule':'smallest 0<alpha<1 retaining >=90% of alpha0-vs-alpha1 median teacher-logit gain and PSNR>alpha0',
    'teacher_gain_retained':float(eligible.sort_values('alpha').iloc[0].teacher_gain_retained),
    'defined_before_cnn_transfer_result':True,
}
(out/'candidate_alpha_rule.json').write_text(json.dumps(rule,indent=2),encoding='utf-8')
print(rule)
display(s[['alpha','teacher_delta_median','psnr_mean','teacher_gain_retained']])


## Train an independent residual CNN on train-only random stegos

The CNN never sees $D_i$, the SRM teacher scores, or the alpha allocation labels during training. Training stegos use only a deterministic random block order at the same fixed net payload.


In [ ]:
def random_order_for_plans(plans,source_id):
    bids=np.asarray([p.block_id for p in plans],dtype=int)
    digest=hashlib.sha256(f'{seed}|{source_id}'.encode()).digest()
    rng=np.random.default_rng(int.from_bytes(digest[:8],'little'))
    out=bids.copy(); rng.shuffle(out)
    return out

def make_random_train_pairs(frame,n_images,bpp):
    covers=[]; stegos=[]; ids=[]
    attempted=min(int(n_images),len(frame))
    for j,row in frame.head(attempted).iterrows():
        sid=str(row.source_id); x=read_gray(row.path)
        plans=analyze_blocks(x,bs)
        order=random_order_for_plans(plans,sid)
        rr=run_frozen_image_precomputed(
            x,sid,bpp,'random',{'random':order},[],bs,seed,False,None,plans=plans
        )
        if rr['feasible']:
            if not (rr['exact_image'] and rr['exact_message'] and rr['ber']==0.0):
                raise RuntimeError(f'Reversibility invariant failed for train source {sid}')
            covers.append(x); stegos.append(rr['stego']); ids.append(sid)
        if (j+1)%250==0:
            print('CNN train pairs',j+1,'/',attempted,'feasible',len(covers))
    return covers,stegos,ids,attempted

N_CNN_TRAIN=min(int(config['detectors']['cnn_train_images']),len(train))
cnn_c,cnn_s,cnn_ids,cnn_attempted=make_random_train_pairs(train,N_CNN_TRAIN,teacher_bpp)
print('CNN training feasible:',len(cnn_c),'/',cnn_attempted, '=',len(cnn_c)/cnn_attempted)
if len(cnn_c)<500:
    raise RuntimeError('Too few feasible train pairs for the independent CNN.')


In [ ]:
cnn,history,device=train_residual_cnn(
    cnn_c,cnn_s,
    validation=None,
    epochs=epochs,
    batch_size=batch_size,
    seed=seed+5100,
)
save_cnn(cnn,models/'residual_cnn_transfer.pt',{
    'history':history,
    'device':device,
    'train_pairs':len(cnn_c),
    'train_attempted':cnn_attempted,
    'payload_bpp':teacher_bpp,
    'allocation':'random',
    'validation_used_for_training':False,
    'seed':seed+5100,
})
pd.DataFrame(history).to_csv(out/'cnn_train_history.csv',index=False)
print('CNN device:',device)
display(pd.DataFrame(history))

# Free the large train pair lists before building the transfer set.
del cnn_c, cnn_s
gc.collect()


## Disjoint validation-half transfer set

`05c` tuned alpha on `val.head(750)`. Here we use validation indices 1000–1999 when the expected 2000-image split is available. The notebook explicitly checks source-ID overlap with the `05c` alpha-tuning set.


In [ ]:
if len(val)>=2000:
    transfer_frame=val.iloc[1000:2000].reset_index(drop=True)
else:
    start=len(val)//2
    transfer_frame=val.iloc[start:].reset_index(drop=True)

alpha_pairs_path=Path('/workspace/results/srm_teacher_risk/alpha_validation_pairs.csv')
if alpha_pairs_path.exists():
    tuned_ids=set(pd.read_csv(alpha_pairs_path).source_id.astype(str).unique())
    transfer_ids=set(transfer_frame.source_id.astype(str))
    overlap=tuned_ids & transfer_ids
    print('05c-tuning / 05d-transfer source-ID overlap:',len(overlap))
    if overlap:
        raise RuntimeError('05d transfer set overlaps the 05c alpha-tuning source images.')

print('Transfer candidates:',len(transfer_frame))
print('Index range in validation: second half')


In [ ]:
def make_orders(block_rows,source_id,alpha):
    bids=np.asarray([r['block_id'] for r in block_rows],int)
    p=np.asarray([r['predictability'] for r in block_rows],float)
    d=np.asarray([r['detectability_risk'] for r in block_rows],float)
    digest=hashlib.sha256(f'{seed}|{source_id}'.encode()).digest()
    rng=np.random.default_rng(int.from_bytes(digest[:8],'little'))
    rnd=bids.copy(); rng.shuffle(rnd)
    return {
        'raster':bids.copy(),
        'random':rnd,
        'predictability':bids[np.argsort(-p,kind='stable')],
        'detectability':bids[np.argsort(d,kind='stable')],
        'joint':bids[rank_blocks(p,d,alpha,1.0-alpha)],
    }

covers=[]
stegos={a:[] for a in alpha_grid}
meta_rows=[]
skipped=[]

for j,row in transfer_frame.iterrows():
    sid=str(row.source_id); x=read_gray(row.path)
    br=local_risk.score_image_blocks(x,sid)
    plans=analyze_blocks(x,bs)
    per_alpha={}
    all_feasible=True
    for a in alpha_grid:
        orders=make_orders(br,sid,a)
        rr=run_frozen_image_precomputed(
            x,sid,teacher_bpp,'joint',orders,br,bs,seed,False,None,plans=plans
        )
        if not rr['feasible']:
            all_feasible=False
            break
        if not (rr['exact_image'] and rr['exact_message'] and rr['ber']==0.0):
            raise RuntimeError(f'Reversibility invariant failed for source {sid}, alpha={a}')
        per_alpha[a]=rr
    if not all_feasible:
        skipped.append(sid)
        continue

    covers.append(x)
    for a in alpha_grid:
        rr=per_alpha[a]
        stegos[a].append(rr['stego'])
        meta_rows.append({
            'source_id':sid,'alpha':a,'psnr':rr['psnr'],'ssim':rr['ssim'],
            'used_blocks':rr['used_blocks'],
            'selected_P_mean':rr['selected_predictability_mean'],
            'selected_D_mean':rr['selected_detectability_risk_mean'],
            'exact_image':rr['exact_image'],'exact_message':rr['exact_message'],'ber':rr['ber'],
        })
    if (j+1)%50==0:
        print('transfer',j+1,'/',len(transfer_frame),'common feasible',len(covers))

common_fraction=len(covers)/max(len(transfer_frame),1)
print('Common-feasible transfer pairs:',len(covers),'/',len(transfer_frame),'=',common_fraction)
print('Skipped because at least one alpha infeasible:',len(skipped))

meta=pd.DataFrame(meta_rows)
meta.to_csv(out/'transfer_common_feasible_metadata.csv',index=False)
(out/'transfer_protocol.json').write_text(json.dumps({
    'payload_bpp':teacher_bpp,
    'alpha_grid':alpha_grid,
    'candidate_alpha':candidate_alpha,
    'attempted_images':len(transfer_frame),
    'common_feasible_images':len(covers),
    'common_feasible_fraction':common_fraction,
    'skipped_source_ids':skipped,
    'validation_subset':'second half; disjoint from 05c alpha tuning',
},indent=2),encoding='utf-8')

if len(covers)<300:
    raise RuntimeError('Too few common-feasible validation images for transfer analysis.')


## Fixed CNN scoring and per-alpha detector metrics

Lower paired score change, lower ROC-AUC (toward 0.5 under the fixed detector orientation), and lower TPR@5% FPR all indicate less transfer to this independent detector. The paired score-change endpoint is primary because every method is evaluated on the same covers.


In [ ]:
cover_scores=score_residual_cnn(cnn,covers,device=device,batch_size=batch_size)
score_map={a:score_residual_cnn(cnn,stegos[a],device=device,batch_size=batch_size) for a in alpha_grid}

rows=[]
for k,a in enumerate(alpha_grid):
    ss=np.asarray(score_map[a],float)
    yy=np.tile([0,1],len(cover_scores))
    scores=np.column_stack([cover_scores,ss]).reshape(-1)
    m=detector_metrics(yy,scores,fixed_fpr=fixed_fpr)
    ci=paired_detector_bootstrap(
        cover_scores,ss,fixed_fpr=fixed_fpr,n_resamples=n_boot,
        confidence=confidence,seed=seed+6000+k,
    )
    delta=ss-cover_scores
    dmean=bootstrap_location_ci(delta,statistic='mean',n_resamples=n_boot,confidence=confidence,seed=seed+6100+k)
    dmed=bootstrap_location_ci(delta,statistic='median',n_resamples=n_boot,confidence=confidence,seed=seed+6200+k)
    rows.append({
        'alpha':a,'n_pairs':len(delta),'auc':m['auc'],
        'auc_ci_low':ci['auc_low'],'auc_ci_high':ci['auc_high'],
        'tpr_at_5pct_fpr':m['tpr_at_fpr'],'tpr_ci_low':ci['tpr_low'],'tpr_ci_high':ci['tpr_high'],
        'cover_score_mean':float(cover_scores.mean()),'stego_score_mean':float(ss.mean()),
        'paired_delta_mean':dmean['estimate'],'paired_delta_mean_ci_low':dmean['low'],'paired_delta_mean_ci_high':dmean['high'],
        'paired_delta_median':dmed['estimate'],'paired_delta_median_ci_low':dmed['low'],'paired_delta_median_ci_high':dmed['high'],
    })

cnn_summary=pd.DataFrame(rows)
cnn_summary.to_csv(out/'cnn_transfer_summary.csv',index=False)
display(cnn_summary)


## Confirmatory paired comparison against predictability-only alpha=1

All bootstrap resamples keep the cover, candidate stego, and reference stego from the same source image together. Negative differences favor the candidate.


In [ ]:
reference_alpha=1.0
pair_rows=[]
for k,a in enumerate(alpha_grid):
    if np.isclose(a,reference_alpha):
        continue
    r=paired_method_bootstrap(
        cover_scores,score_map[a],score_map[reference_alpha],
        fixed_fpr=fixed_fpr,n_resamples=max(n_boot,5000),confidence=confidence,
        seed=seed+7000+k,
    )
    r['alpha']=a; r['reference_alpha']=reference_alpha
    pair_rows.append(r)

paired=pd.DataFrame(pair_rows).sort_values('alpha').reset_index(drop=True)
paired.to_csv(out/'paired_vs_alpha1.csv',index=False)
display(paired[[
    'alpha','n_pairs','delta_mean_diff','delta_mean_diff_low','delta_mean_diff_high',
    'auc_diff','auc_diff_low','auc_diff_high','tpr_diff','tpr_diff_low','tpr_diff_high'
]])


In [ ]:
fig,ax=plt.subplots(figsize=(6.2,4.2))
ax.plot(cnn_summary.alpha,cnn_summary.auc,marker='o')
ax.axhline(0.5,linewidth=1)
ax.set_xlabel(r'$\alpha$')
ax.set_ylabel('Residual-CNN ROC-AUC')
ax.set_title('Independent detector transfer across allocator weights')
ax.grid(True,alpha=.2); fig.tight_layout()
fig.savefig(out/'alpha_vs_cnn_auc.png',dpi=300); plt.show()

fig,ax=plt.subplots(figsize=(6.2,4.2))
ax.plot(cnn_summary.alpha,cnn_summary.paired_delta_mean,marker='o')
ax.axhline(0,linewidth=1)
ax.set_xlabel(r'$\alpha$')
ax.set_ylabel('Mean paired CNN score change')
ax.set_title('Independent detector score change across allocator weights')
ax.grid(True,alpha=.2); fig.tight_layout()
fig.savefig(out/'alpha_vs_cnn_delta.png',dpi=300); plt.show()


## Decision checkpoint

The **primary confirmatory criterion** was fixed above: for the `05c` candidate alpha, the 95% paired bootstrap CI for

\[
[ s_{CNN}(Y_{candidate})-s_{CNN}(X)]-[s_{CNN}(Y_{\alpha=1})-s_{CNN}(X)]
\]

must lie fully below zero.

AUC and TPR differences are secondary transfer evidence. No test-set result is consumed here.


In [ ]:
cand=paired.loc[np.isclose(paired.alpha,candidate_alpha)].iloc[0]
cand_summary=cnn_summary.loc[np.isclose(cnn_summary.alpha,candidate_alpha)].iloc[0]
ref_summary=cnn_summary.loc[np.isclose(cnn_summary.alpha,1.0)].iloc[0]

primary_pass=bool(cand.delta_mean_diff_high < 0.0)
point_direction=bool(cand.delta_mean_diff < 0.0)
auc_direction=bool(cand.auc_diff < 0.0)
tpr_direction=bool(cand.tpr_diff < 0.0)

if primary_pass:
    decision='TRANSFER_CONFIRMED_PRIMARY_ENDPOINT'
elif point_direction:
    decision='TRANSFER_SIGNAL_NOT_CONCLUSIVE'
else:
    decision='TRANSFER_NOT_CONFIRMED'

decision_obj={
    'decision':decision,
    'candidate_alpha':candidate_alpha,
    'reference_alpha':1.0,
    'payload_bpp':teacher_bpp,
    'primary_endpoint':'paired CNN score-change difference candidate minus alpha=1.0',
    'primary_difference':float(cand.delta_mean_diff),
    'primary_ci_low':float(cand.delta_mean_diff_low),
    'primary_ci_high':float(cand.delta_mean_diff_high),
    'primary_pass':primary_pass,
    'auc_candidate':float(cand_summary.auc),
    'auc_reference':float(ref_summary.auc),
    'auc_diff':float(cand.auc_diff),
    'auc_direction_supports_candidate':auc_direction,
    'tpr_candidate':float(cand_summary.tpr_at_5pct_fpr),
    'tpr_reference':float(ref_summary.tpr_at_5pct_fpr),
    'tpr_diff':float(cand.tpr_diff),
    'tpr_direction_supports_candidate':tpr_direction,
    'common_feasible_pairs':int(len(covers)),
    'test_split_used':False,
    'frozen_allocator_modified':False,
}
(out/'decision.json').write_text(json.dumps(decision_obj,indent=2),encoding='utf-8')
print(json.dumps(decision_obj,indent=2))

if primary_pass:
    print('\nNEXT: candidate alpha passed the independent-detector primary endpoint.')
    print('Do not run test yet; review secondary metrics and then freeze alpha in a separate step.')
elif point_direction:
    print('\nNEXT: direction is favorable but CI crosses zero. Do not freeze; inspect power / sample size / detector behavior.')
else:
    print('\nNEXT: transfer was not confirmed. Do not freeze the allocator and do not run the test experiment.')


### Outputs

Saved under `/workspace/results/cnn_transfer/`:

- `candidate_alpha_rule.json` — alpha rule fixed from 05c before CNN transfer;
- `cnn_train_history.csv` — train-only CNN optimization history;
- `transfer_protocol.json` — disjoint validation subset and common-feasible accounting;
- `transfer_common_feasible_metadata.csv` — per-image/per-alpha RDH metrics;
- `cnn_transfer_summary.csv` — AUC, TPR@5% FPR and paired score-change intervals;
- `paired_vs_alpha1.csv` — paired bootstrap method differences against predictability-only allocation;
- `decision.json` — locked primary-endpoint decision;
- two publication-oriented diagnostic figures.

If the primary endpoint is confirmed, the next notebook should freeze the allocator **without revisiting the test split**.
